# Sandbox: Nicolas

Conflict, Democracy and Arms project. This is your personal notebook for trying things out.
Only Nicolas edits this file, so it never causes a merge conflict with Yair.

**How to use it**

1. Pull first (GitHub Desktop: Fetch origin, then Pull).
2. Work through the tasks in order. Every step has its own empty cell. Write your own code there.
3. Open a hint only when you are stuck. Hint 1 is a small nudge, later hints say more.
4. Run the checkpoint after each task. OK means done.
5. When a task is done, paste the working code into the main notebook listed below and push.

| Part | Task | Paste into |
| --- | --- | --- |
| Practice | How a merge works, on toy data | nothing, just for learning |
| A | V-Dem: democracy and regime per country-year | `03_vdem.ipynb` |
| B | UCDP: conflict per country-year and conflict periods | `02_ucdp.ipynb` |
| C | Descriptives: conflict by regime type | `06_analysis.ipynb`, Part A |
| D | Chart functions: regime bands, conflict bars | `07_charts.ipynb`, then `app/panels_conflict.py` |

Variable, column and file names come from `CONVENTIONS.md`. Use them exactly, otherwise the merge in 05 breaks.

## How we write code in this project

We want code we understand completely, so we write it slowly and in small pieces. More lines are fine.

1. **One operation per line.** No long chains like `df.groupby(...).agg(...).reset_index().rename(...)`. Do each step on its own line and give the result a name.
2. **Look after every step.** Print `.shape`, `.head()` or a few values after each line that changes the data. If a number looks strange, stop and find out why.
3. **Simple tools first.** Filtering with conditions, `rename`, `sort_values`, dictionaries, lists, `for` loops and `if` statements. A loop that is slow but clear is better than one line we cannot explain.
4. **Comment in your own words.** Above each block, write one comment that says what it does.
5. **Use a new function only after trying it on a tiny example.** For example, before merging two big tables, merge two tables with three rows each (see the practice section).
6. **Use AI only for hints and explanations**, never to write the solution. Ask: "explain this error" or "what does this function do", and say that you want simple step-by-step code.

Tools we use: `read_csv`, `read_excel`, `head`, `shape`, `columns`, `isna`, `sum`, `rename`, `sort_values`, conditions with `&` and `|`, `isin`, `str.endswith`, `str.split`, `astype`, `for` loops, `iterrows`, dictionaries, lists, `pd.DataFrame(list_of_dicts)`, `unique`, `value_counts`, `between`, `notna`, `fillna`, `groupby` with one function at a time, `reset_index`, `nunique`, `merge` (after the practice), `concat`, `.copy()`, `to_csv`, and plotly express.

If you see a `SettingWithCopyWarning`, add `.copy()` after the filter that made the table, for example `df = panel[panel["iso3"] == "UKR"].copy()`.

## Setup

This cell only loads the packages and sets the folder paths. Run it every time you open the notebook.

In [ ]:
import pandas as pd
import plotly.express as px

RAW = "../data/raw/"
CLEAN = "../data/clean/"

pd.set_option("display.max_columns", 50)
print("pandas", pd.__version__)

import os

---
## Practice: how a merge works (do this before any real merge)

A merge puts two tables side by side, matching rows that have the same value in a key column. We never did this in class, so we try it on tiny tables first, where we can check every row by eye.

Run the cell below. It creates two small tables.

In [ ]:
# Toy data: three countries with a democracy score, and arms imports for some of them
demo = pd.DataFrame({
    "iso3": ["CHE", "IND", "UKR"],
    "year": [2000, 2000, 2000],
    "democracy_index": [0.90, 0.70, 0.45],
})
arms = pd.DataFrame({
    "iso3": ["IND", "UKR", "SAU"],
    "year": [2000, 2000, 2000],
    "arms_imports_tiv": [900, 20, 1500],
})
print(demo)
print(arms)

**Step 1. Predict first.** Before you run anything, write in a comment: which countries appear in both tables? Which only in one?

In [ ]:
# your prediction here

**Step 2. Inner merge.** Merge `demo` and `arms` on `iso3` and `year` with `how="inner"`. Call the result `m_inner`. Print it. How many rows? Which country is missing and why?

In [ ]:
# your code here

**Step 3. Left merge.** Same, with `how="left"` and `demo` on the left. Call it `m_left`. Print it. What appears in `arms_imports_tiv` for CHE? Why?

In [ ]:
# your code here

**Step 4. Fill the gap.** In our project, a country with no arms deliveries imported nothing. Replace the missing value in `m_left` with 0. Print again. Also print `m_left.dtypes`: why is `arms_imports_tiv` now a decimal number (float)?

In [ ]:
# your code here

**Step 5. The danger: duplicate keys.** Run the cell below. It adds IND a second time to `arms`. Merge `demo` with `arms_dup` (left merge), call the result `m_dup` and count the rows. What happened to IND?

In [ ]:
arms_dup = pd.concat([arms, pd.DataFrame({"iso3": ["IND"], "year": [2000], "arms_imports_tiv": [5]})])
print(arms_dup)

In [ ]:
# your code here

**Step 6. Write the two rules in your own words** (as a comment): what does `how="left"` keep, and what must you check about the right table before every merge?

In [ ]:
# your rules here

<details><summary><b>Hint 1</b> (open only if stuck)</summary>

The function is `pd.merge(left_table, right_table, on=[...], how=...)`. `on` is a list of the key columns, `how` is the kind of merge as text.

</details>

<details><summary><b>Hint 2</b> (open only if stuck)</summary>

`pd.merge(demo, arms, on=["iso3", "year"], how="inner")`. For the fill: `m_left["arms_imports_tiv"] = m_left["arms_imports_tiv"].fillna(0)`.
Rule of thumb for every real merge: print the number of rows before and after. With a left merge, the number must stay the same. If it grows, the right table has duplicate keys.

</details>

**Checkpoint.**

In [ ]:
assert len(m_inner) == 2, "inner keeps only countries in both tables"
assert len(m_left) == 3, "left keeps every row of demo"
assert sorted(m_left["iso3"]) == ["CHE", "IND", "UKR"], "left merge keeps the rows of the LEFT table (demo)"
assert m_left["arms_imports_tiv"].isna().sum() == 0, "fill the missing value with 0"
assert m_left.loc[m_left["iso3"] == "CHE", "arms_imports_tiv"].sum() == 0
assert len(m_dup) == 4, "the duplicate IND row in arms_dup creates an extra row"
print("OK")

---
## Part A: V-Dem (03)

Do this first. 01 Crosswalk and 05 Merge both need `vdem.csv`.

### A1 Load only the columns you need

**Goal:** a dataframe `vdem_raw` with six columns.

**Step 1.** The V-Dem file has more than 4,000 columns and is several hundred MB. Make a list called `cols` with the six names we need: `country_name`, `country_text_id`, `COWcode`, `year`, `v2x_polyarchy`, `v2x_regime`.

In [ ]:
# your code here

**Step 2.** Load the file with `pd.read_csv` and the argument `usecols=cols`. The path is `RAW + "vdem/V-Dem-CY-Core-v16.csv"`.

In [ ]:
# your code here

**Step 3.** Look at it: print `vdem_raw.shape`, then `vdem_raw.head()`.

In [ ]:
# your code here

**Step 4.** Count missing values per column with `vdem_raw.isna().sum()`. Write in a comment which columns have missing values.

In [ ]:
# your code here

<details><summary><b>Hint</b> (open only if stuck)</summary>

`usecols` tells pandas to read only these columns, so loading is fast. Everything else works like a normal `read_csv`.

</details>

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert "vdem_raw" in globals(), "create a dataframe called vdem_raw"
assert vdem_raw.shape[1] == 6, "vdem_raw should have exactly 6 columns"
assert vdem_raw["year"].min() < 1950
print("OK")

**In your own words** (one or two sentences, write them here): what did this task do, and why do we need it?

### A2 Filter years and rename

**Goal:** a dataframe `vdem` with years 1950 to 2025 and the column names from CONVENTIONS.md.

**Step 1.** Make a True/False condition for `year >= 1950` (call it `from_1950`) and one for `year <= 2025` (call it `until_2025`). Print how many rows are True for each (`.sum()`).

In [ ]:
# your code here

**Step 2.** Combine both with `&` and use it to filter. Call the result `vdem`. Add `.copy()` at the end of the filter line. Print the shape before and after.

In [ ]:
# your code here

**Step 3.** Write a dictionary `new_names` with old name → new name: `country_text_id` → `iso3`, `COWcode` → `cow_code`, `v2x_polyarchy` → `democracy_index`, `v2x_regime` → `regime_type`. Use it with `rename(columns=new_names)`. Without `columns=` nothing gets renamed and you get no error, so check `vdem.columns` afterwards.

In [ ]:
# your code here

**Step 4.** Find the rows where `regime_type` is missing: `vdem[vdem["regime_type"].isna()]`. Look at them. Decide: drop them or keep them? Dropping a country-year here also drops its conflicts and arms imports in the merge (05), so write the decision and the number of rows in the decisions table, then do it.

In [ ]:
# your code here

**Step 5.** Convert `regime_type` to whole numbers with `astype(int)`. Print `vdem["regime_type"].unique()`.

In [ ]:
# your code here

<details><summary><b>Hint 1</b> (open only if stuck)</summary>

Each condition is its own line: `from_1950 = vdem_raw["year"] >= 1950`. Then `vdem = vdem_raw[from_1950 & until_2025].copy()`.

</details>

<details><summary><b>Hint 2</b> (open only if stuck)</summary>

`astype(int)` fails if a value is still missing. That is why step 4 comes first. To drop: keep only rows where the value is not missing, `vdem = vdem[vdem["regime_type"].notna()]`.

</details>

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert "vdem" in globals()
for col in ["iso3", "country_name", "cow_code", "year", "democracy_index", "regime_type"]:
    assert col in vdem.columns, f"missing column {col}"
assert vdem["year"].between(1950, 2025).all()
assert vdem["democracy_index"].dropna().between(0, 1).all()
assert vdem["regime_type"].dtype.kind == "i", "regime_type must be an integer (astype(int))"
assert set(vdem["regime_type"].unique()) <= {0, 1, 2, 3}
assert vdem["iso3"].isna().sum() == 0
assert vdem.duplicated(["iso3", "year"]).sum() == 0, "one row per iso3-year"
print("OK")

**In your own words** (one or two sentences, write them here): what did this task do, and why do we need it?

### A3 Regime labels

**Goal:** a column `regime_label` with readable names, spelled exactly as below (the charts and the app depend on it).

**Step 1.** Write a dictionary `labels` with the four numbers as keys: 0 = `closed autocracy`, 1 = `electoral autocracy`, 2 = `electoral democracy`, 3 = `liberal democracy` (all lower case). Test it: `labels[2]` should print electoral democracy.

In [ ]:
# your code here

**Step 2.** Make the new column. Simple way: an empty list, a `for` loop over `vdem["regime_type"]`, append `labels[value]` for each value, then `vdem["regime_label"] = the_list`. (Once that works, you may try `vdem["regime_type"].map(labels)` and check it gives the same result.)

In [ ]:
# your code here

**Step 3.** Count how often each label appears with `vdem["regime_label"].value_counts()`.

In [ ]:
# your code here

<details><summary><b>Hint</b> (open only if stuck)</summary>

The list must have exactly as many entries as `vdem` has rows. Print `len(the_list)` and `len(vdem)` before you assign it.

</details>

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert vdem["regime_label"].isna().sum() == 0
assert set(vdem["regime_label"]) == {"closed autocracy", "electoral autocracy", "electoral democracy", "liberal democracy"}, "check spelling and lower case"
assert (vdem[vdem["regime_type"] == 3]["regime_label"] == "liberal democracy").all()
assert (vdem[vdem["regime_type"] == 0]["regime_label"] == "closed autocracy").all()
print("OK")

**In your own words** (one or two sentences, write them here): what did this task do, and why do we need it?

### A4 Regime periods for the timeline

**Goal:** `vdem_regime_spans`: one row per uninterrupted regime period, with exactly the columns `iso3`, `regime_label`, `start`, `end`.

**Step 1.** Sort `vdem` by `iso3` and then `year` with `sort_values(["iso3", "year"])`. Print the rows of one country (for example `"UKR"`) and write down by eye where the regime changes.

In [ ]:
# your code here

**Step 2.** Write the loop for **one country only** (`"UKR"`). A period ends when the regime label changes, and also when a year is missing (the country did not exist for some years). Each finished period is a dictionary with the keys `iso3`, `regime_label`, `start`, `end`. Print the list and compare it with what you wrote down in step 1.

In [ ]:
# your code here

**Step 3.** When it is correct for Ukraine, run it for all countries and turn the list into the dataframe `vdem_regime_spans` with `pd.DataFrame(periods)`. Print the number of rows and the rows for `"CHE"`.

In [ ]:
# your code here

<details><summary><b>Hint 1</b> (open only if stuck)</summary>

Think of it like reading a country's history year by year and writing down a new line on paper every time a period ends. You need: a list `periods` to collect the finished periods, and a few variables you keep updating while you go through the years (start year, previous year, and what you track for the period).

</details>

<details><summary><b>Hint 2</b> (open only if stuck)</summary>

Plan in plain words (turn each line into code yourself):

```
periods = empty list
for each country code in the list of countries:
    take the rows of this country, sorted by year
    prev_year = None
    for each row of this country:
        if prev_year is None:                      (first row of this country)
            start = this year, current_label = this label
        elif (your condition: when does a period end? two cases, see step 2):
            add {"iso3": code, "regime_label": current_label, "start": start, "end": prev_year} to periods
            start = this year, current_label = this label
        prev_year = this year
    after the loop over the rows: add the last period too (people often forget this one)
vdem_regime_spans = pd.DataFrame(periods)
```

</details>

<details><summary><b>Hint 3</b> (open only if stuck)</summary>

Useful pieces: `vdem["iso3"].unique()` gives the list of countries. `country_rows = vdem[vdem["iso3"] == code]` takes one country. `for index, row in country_rows.iterrows():` loops over its rows, and `row["year"]` reads one value. `periods.append({...})` adds one period.

</details>

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert list(vdem_regime_spans.columns) == ["iso3", "regime_label", "start", "end"]
assert (vdem_regime_spans["end"] >= vdem_regime_spans["start"]).all()
years_covered = (vdem_regime_spans["end"] - vdem_regime_spans["start"] + 1).sum()
assert years_covered == len(vdem), f"periods cover {years_covered} years, vdem has {len(vdem)} rows (forgot the last period, or bridged a gap?)"
print("OK")

**In your own words** (one or two sentences, write them here): what did this task do, and why do we need it?

### A5 Save

**Goal:** the two clean files in `data/clean/`.

**Step 1.** Save `vdem` as `vdem.csv` and `vdem_regime_spans` as `vdem_regime_spans.csv`. Use `index=False` so pandas does not add an extra column.

In [ ]:
# your code here

<details><summary><b>Hint</b> (open only if stuck)</summary>

`vdem.to_csv(CLEAN + "vdem.csv", index=False)`

</details>

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert os.path.exists(CLEAN + "vdem.csv") and os.path.exists(CLEAN + "vdem_regime_spans.csv")
assert "Unnamed: 0" not in pd.read_csv(CLEAN + "vdem.csv").columns, "save with index=False"
print("OK")

**In your own words** (one or two sentences, write them here): what did this task do, and why do we need it?

---
## Part B: UCDP (02)

Needs `crosswalk_gw.csv` from 01 Crosswalk and the merge practice above.

### B1 Load and inspect

**Goal:** `ucdp_raw` loaded and understood.

**Step 1.** Load `UcdpPrioConflict_v26_1.xlsx` with `pd.read_excel`. Print shape and the column names.

In [ ]:
# your code here

**Step 2.** Print the four columns we need for the first 10 rows: `conflict_id`, `year`, `gwno_loc`, `intensity_level`. Write in a comment what one row stands for.

In [ ]:
# your code here

**Step 3.** Look for rows where `gwno_loc` contains a comma. Convert the column to text first (`astype(str)`), then use `str.contains(",")`. How many are there? Print five of them. Why can one conflict-year have several countries?

In [ ]:
# your code here

**Step 4.** Keep only rows with `year >= 1950` and overwrite `ucdp_raw` (add `.copy()`). Our panel starts in 1950, and 01 Crosswalk only matches codes from 1950 on. Print how many rows were removed.

In [ ]:
# your code here

<details><summary><b>Hint</b> (open only if stuck)</summary>

One row = one conflict in one year. Interstate wars involve several countries, so UCDP lists all their codes in one cell, for example "651, 666".

</details>

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert "ucdp_raw" in globals()
for col in ["conflict_id", "year", "gwno_loc", "intensity_level"]:
    assert col in ucdp_raw.columns
assert ucdp_raw["year"].min() >= 1950, "keep only years from 1950 on (step 4)"
print("OK")

**In your own words** (one or two sentences, write them here): what did this task do, and why do we need it?

### B2 One row per country per conflict-year

**Goal:** `ucdp_long`: like `ucdp_raw`, but a row with "651, 666" becomes two rows, one with `gw_code` 651 and one with 666.

**Step 1.** Try the idea on one value: take the text `"651, 666"`, split it with `.split(",")`, print the list. Then loop over the list, remove spaces with `.strip()` and turn each piece into a number with `int()`. Print each number.

In [ ]:
# your code here

**Step 2.** Now the loop over the table: make an empty list `rows`. Loop over `ucdp_raw` with `iterrows()`. For each row, split `str(row["gwno_loc"])` as above, and for every code append one dictionary `{"conflict_id": ..., "year": ..., "gw_code": ..., "intensity_level": ...}` to `rows`.

In [ ]:
# your code here

**Step 3.** Turn the list into a dataframe: `ucdp_long = pd.DataFrame(rows)`. Print the number of rows of `ucdp_raw` and of `ucdp_long`. By how many did it grow? Does that match the number of commas you found in B1?

In [ ]:
# your code here

<details><summary><b>Hint 1</b> (open only if stuck)</summary>

Step 1 is plain Python, no pandas: `text = "651, 666"`, `parts = text.split(",")`, then `for p in parts: print(int(p.strip()))`.

</details>

<details><summary><b>Hint 2</b> (open only if stuck)</summary>

Inside the big loop you have two loops: the outer one over the rows of the table, the inner one over the codes in that row. Every time the inner loop runs, one dictionary goes into `rows`.

</details>

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert list(ucdp_long.columns) == ["conflict_id", "year", "gw_code", "intensity_level"]
assert ucdp_long["gw_code"].dtype.kind == "i", "gw_code must be an integer"
expected = len(ucdp_raw) + ucdp_raw["gwno_loc"].astype(str).str.count(",").sum()
assert len(ucdp_long) == expected, f"expected {expected} rows: one per code"
print("OK")

**In your own words** (one or two sentences, write them here): what did this task do, and why do we need it?

### B3 Add iso3 with a merge

**Goal:** every row in `ucdp_long` gets an `iso3`, and no row is lost or added.

**Step 1.** Load `crosswalk_gw.csv` into `crosswalk_gw`. Print its first rows. Which column is the key that also exists in `ucdp_long`?

In [ ]:
# your code here

**Step 2.** Before the merge: check that every `gw_code` appears only once in the crosswalk (`crosswalk_gw["gw_code"].duplicated().sum()` must be 0). Save the length: `n_before = len(ucdp_long)`.

In [ ]:
# your code here

**Step 3.** Merge with `how="left"`, `ucdp_long` on the left, and overwrite `ucdp_long` with the result. Print the length again. It must equal `n_before`.

In [ ]:
# your code here

**Step 4.** List the codes that got no `iso3`: filter the rows where `iso3` is missing and print the unique `gw_code` values. Those need a fix in the crosswalk (01), not here. Rows without iso3 would disappear silently in B4.

In [ ]:
# your code here

<details><summary><b>Hint</b> (open only if stuck)</summary>

Exactly like the practice: `pd.merge(ucdp_long, crosswalk_gw, on="gw_code", how="left")`. If the row count changed, go back to step 2.

</details>

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert len(ucdp_long) == n_before, "merge added rows: duplicate gw_code in crosswalk_gw"
assert ucdp_long["iso3"].isna().sum() == 0, "some codes have no iso3: fix crosswalk_gw in 01"
print("OK")

**In your own words** (one or two sentences, write them here): what did this task do, and why do we need it?

### B4 Country-year table

**Goal:** `ucdp_cy` with one row per `iso3` and `year`, and the columns `iso3`, `year`, `n_conflicts`, `max_intensity`, `conflict_active`.

**Step 1.** Count conflicts: group by `["iso3", "year"]` and take `["conflict_id"].nunique()`. Call it `counts`. On the next line use `reset_index()` to turn it back into a normal table. On the next line rename the column to `n_conflicts`. Print the first rows.

In [ ]:
# your code here

**Step 2.** Highest intensity: same grouping, but `["intensity_level"].max()`. Call it `highest`, reset the index, rename to `max_intensity`.

In [ ]:
# your code here

**Step 3.** Put the two tables together with a merge on `iso3` and `year`. Both have exactly the same rows, so the row count must not change. Call it `ucdp_cy`.

In [ ]:
# your code here

**Step 4.** Add the column `conflict_active` with the value 1 for every row (`ucdp_cy["conflict_active"] = 1`). Spot check: print the rows for `"UKR"` from 2014 on.

In [ ]:
# your code here

<details><summary><b>Hint 1</b> (open only if stuck)</summary>

We do two separate small groupbys and merge them, so each line does one thing. `nunique` counts different conflicts, so a conflict listed twice for the same country-year still counts once.

</details>

<details><summary><b>Hint 2</b> (open only if stuck)</summary>

If groupby feels like magic, do it once by hand for one country-year: filter `ucdp_long` to UKR 2015, count the different `conflict_id` values with `len(set(...))`, take the max of `intensity_level`. Your groupby result must show the same numbers for UKR 2015.

</details>

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert list(ucdp_cy.columns) == ["iso3", "year", "n_conflicts", "max_intensity", "conflict_active"]
assert ucdp_cy.duplicated(["iso3", "year"]).sum() == 0
assert set(ucdp_cy["max_intensity"].unique()) <= {1, 2}
assert (ucdp_cy["conflict_active"] == 1).all()
print("OK")

**In your own words** (one or two sentences, write them here): what did this task do, and why do we need it?

### B5 Conflict periods and save

**Goal:** `ucdp_spans` (conflict periods for the timeline bars) with exactly the columns `iso3`, `start`, `end`, `max_intensity`, and both CSV files saved.

**Step 1.** Sort `ucdp_cy` by `iso3` and `year`.

In [ ]:
# your code here

**Step 2.** Loop for **one country first** (`"COL"`). A new period starts only when the year is **not** the previous year + 1 (a gap without conflict). A change in intensity does not start a new period. While a period runs, keep its highest intensity in `current_max`.

In [ ]:
# your code here

**Step 3.** Run it for all countries, make `ucdp_spans` with `pd.DataFrame(periods)`, and save `ucdp_cy.csv` and `ucdp_spans.csv` with `index=False`.

In [ ]:
# your code here

<details><summary><b>Hint 1</b> (open only if stuck)</summary>

Think of it like reading a country's history year by year and writing down a new line on paper every time a period ends. You need: a list `periods` to collect the finished periods, and a few variables you keep updating while you go through the years (start year, previous year, and what you track for the period).

</details>

<details><summary><b>Hint 2</b> (open only if stuck)</summary>

Plan in plain words:

```
periods = empty list
for each country code:
    rows of this country, sorted by year
    prev_year = None
    for each row:
        if prev_year is None:                 (first row)
            start = this year, current_max = this max_intensity
        elif (your condition: when is there a gap?):
            add {"iso3": code, "start": start, "end": prev_year, "max_intensity": current_max} to periods
            start = this year, current_max = this max_intensity
        else:                                 (the period continues)
            current_max = max(current_max, this max_intensity)
        prev_year = this year
    after the loop: add the last period
```

</details>

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert list(ucdp_spans.columns) == ["iso3", "start", "end", "max_intensity"]
assert (ucdp_spans["end"] >= ucdp_spans["start"]).all()
assert (ucdp_spans["end"] - ucdp_spans["start"] + 1).sum() == len(ucdp_cy), "spans must cover every conflict year once"
keys = set(zip(ucdp_cy["iso3"], ucdp_cy["year"]))
n_starts = sum(1 for (cc, y) in keys if (cc, y - 1) not in keys)
assert len(ucdp_spans) == n_starts, "one span per uninterrupted run of conflict years"
assert set(ucdp_spans["max_intensity"]) <= {1, 2}
assert os.path.exists(CLEAN + "ucdp_cy.csv") and os.path.exists(CLEAN + "ucdp_spans.csv")
print("OK")

**In your own words** (one or two sentences, write them here): what did this task do, and why do we need it?

---
## Part C: Descriptives (06 Part A)

Needs `panel.csv` from 05 Merge.

### C1 Conflict by regime type

**Goal:** a dictionary `conflict_share` with the share of country-years in conflict for each regime label.

**Step 1.** Load `panel.csv` into `panel`.

In [ ]:
# your code here

**Step 2.** Do it by hand for one label first: filter the rows with `regime_label == "closed autocracy"`, count how many there are, count how many of those have `conflict_active == 1`, divide. Print the share.

In [ ]:
# your code here

**Step 3.** Now for all four labels with a `for` loop over the labels. Store each result in a dictionary `conflict_share = {label: share}` and print it. Then check the same numbers with `panel.groupby("regime_label")["conflict_active"].mean()`.

In [ ]:
# your code here

**Step 4.** Write one sentence with the result in a comment (or turn the cell into a markdown cell: in VS Code press M on the selected cell).

In [ ]:
# your code here

<details><summary><b>Hint</b> (open only if stuck)</summary>

The mean of a 0/1 column is exactly the share of 1s. That is why step 3 gives the same numbers as your loop.

</details>

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
assert len(conflict_share) == 4
for label in conflict_share:
    expected = panel[panel["regime_label"] == label]["conflict_active"].mean()
    assert abs(conflict_share[label] - expected) < 1e-9, f"share for {label} is not right"
print("OK")

**In your own words** (one or two sentences, write them here): what did this task do, and why do we need it?

---
## Part D: Chart functions (07)

Each function takes a dataframe and returns a plotly figure. Test it here, then copy it unchanged into `07_charts.ipynb` and from there into `app/panels_conflict.py`.

**Rules for every timeline:** x-axis 1950 to 2025, height 180, margins `dict(l=60, r=10, t=30, b=10)` with `fig.update_yaxes(automargin=False)`, the legend above the chart, a short chart title, a y-axis title with the unit, no x-axis title, and a dashed vertical line at `year` if one is passed. Then all timelines line up when they are stacked in the app (see `CONVENTIONS.md`).

Build each function in two rounds: first write the code **without** a function for one country and look at the chart. Only when the chart is right, put the same lines into a `def`, replace `"UKR"` with the argument `iso3`, and use only the arguments inside (never a global table).

### D0 Load the clean files and agree on colours

**Goal:** the chart data loaded from the CSV files (exactly as the app will do it), and one colour dictionary.

**Step 1.** Load `vdem_regime_spans.csv` and `ucdp_spans.csv` from `CLEAN` (`panel` is already loaded from Part C).

In [ ]:
# your code here

**Step 2.** Write a dictionary `regime_colours` with one colour (hex code, for example `"#1D4E89"`) for each of the four regime labels, and a dictionary `intensity_colours` for `"minor"` and `"war"`. Agree on them with Yair; they go to the top of `07_charts.ipynb` and of all three panels files.

In [ ]:
# your code here

**In your own words** (one or two sentences, write them here): what did this task do, and why do we need it?

### D1 regime_bands(spans, iso3, year=None)

**Goal:** coloured bands, one per regime period, all on one line.

**Step 1.** Without a function: filter `vdem_regime_spans` to `"UKR"` and add `.copy()`. Add a column `length` = `end - start + 1`. Print the table.

In [ ]:
# your code here

**Step 2.** Draw it with `px.bar`: `x="length"`, `base="start"`, `y="iso3"`, `color="regime_label"`, `orientation="h"`, `color_discrete_map=regime_colours`. Without `y="iso3"` every period gets its own row.

In [ ]:
# your code here

**Step 3.** Make it look like the other timelines: `fig.update_xaxes(range=[1950, 2025])`, `fig.update_layout(height=180, margin=dict(l=60, r=10, t=30, b=10), xaxis_title="", yaxis_title="", title="Regime", legend=dict(orientation="h", y=1.3))` and `fig.update_yaxes(automargin=False)` (otherwise labels of different widths push the plot areas apart).

In [ ]:
# your code here

**Step 4.** If `year` is given, add `fig.add_vline(x=year, line_dash="dash")`. Use an `if year is not None:` block.

In [ ]:
# your code here

**Step 5.** Now wrap the working lines in `def regime_bands(spans, iso3, year=None):` and end it with `return fig`.

In [ ]:
# your code here

<details><summary><b>Hint</b> (open only if stuck)</summary>

`base` tells plotly where each bar starts; `x` is how long it is. So a period from 1994 to 2004 gets base 1994 and length 11.

</details>

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
test = regime_bands(vdem_regime_spans, "CHE")
n_periods = len(vdem_regime_spans[vdem_regime_spans["iso3"] == "CHE"])
assert sum(len(t.x) for t in test.data) == n_periods, "the function must use its arguments, not UKR"
only_che = vdem_regime_spans[vdem_regime_spans["iso3"] == "CHE"]
assert sum(len(t.x) for t in regime_bands(only_che, "UKR").data) == 0, "filter the argument spans, not the global table"
assert len(test.layout.shapes) == 0, "no dashed line when year is None"
assert len(regime_bands(vdem_regime_spans, "CHE", year=2000).layout.shapes) == 1
regime_bands(vdem_regime_spans, "UKR", year=2014).show()
print("OK")

**In your own words** (one or two sentences, write them here): what did this task do, and why do we need it?

### D2 conflict_bars(spans, iso3, year=None)

**Goal:** bars for each conflict period, coloured by intensity, same look as D1.

**Step 1.** `max_intensity` is a number, so plotly would draw a colour scale. Make a text column `intensity_label` first, with a dictionary `{1: "minor", 2: "war"}` and a loop (like in A3). Colour by it with `color_discrete_map=intensity_colours`.

In [ ]:
# your code here

**Step 2.** Same bar structure and layout settings as D1.

In [ ]:
# your code here

**Step 3.** Handle a country without conflict: after filtering, `if len(df) == 0:` make an empty figure (`fig = px.bar()`), add `fig.add_annotation(text="No conflict recorded", showarrow=False)`, set the same height, and return it.

In [ ]:
# your code here

**Step 4.** Wrap it in the function.

In [ ]:
# your code here

**Checkpoint.** Run this cell. If it prints OK, the task is done. Do not change it.

In [ ]:
only_col = ucdp_spans[ucdp_spans["iso3"] == "COL"]
assert len(conflict_bars(only_col, "UKR").layout.annotations) == 1, "filter the argument spans, not the global table"
empty = conflict_bars(ucdp_spans, "XXX")
assert empty is not None, "return the empty figure"
assert len(empty.layout.annotations) == 1, "a country without conflict shows one text annotation"
conflict_bars(ucdp_spans, "COL", year=1990).show()
print("OK")

**In your own words** (one or two sentences, write them here): what did this task do, and why do we need it?

## Before you move code into the main notebook

- [ ] Kernel → Restart and Run All runs this sandbox from top to bottom without an error
- [ ] All checkpoints above print OK
- [ ] Names follow `CONVENTIONS.md`
- [ ] You wrote the "in your own words" sentence for every task
- [ ] You can explain every line to your partner without looking at the hints
- [ ] Copy only the working cells into the numbered main notebook (no experiments, no long printouts)
- [ ] Run the main notebook with Restart and Run All, then commit and push